# SIH26158 - single-pass drone video to 3D

The problem statement's mandatory input is a **video**, and its first Key Challenge is
*"limited viewing angles due to single flight path."* This notebook runs that case end to
end: one continuous pass from a real drone video file, through a learned multi-view model,
to a filtered point cloud and a mesh.

## Why the footage changed

The previous run used 40 s over Nicosia and produced a smear. The cause was not the
filtering - it was the input, and nothing in S1 was measuring the properties that mattered:

| Property | Nicosia | Why it is fatal |
|---|---|---|
| Sky, whole frame | **52%** | Half of every view carries no geometry at all |
| Horizon in frame | **100% of frames** | Depth spans ~100 m to ~30 km; no feed-forward model resolves 2.5 orders of magnitude in one point map |
| Shots in the clip | **2** | A 180-frame cut at keyframe 11. Two unrelated trajectories fed to a model that assumes one |
| Burned-in watermark | 0.1% of pixels | Static in *image* space across every view, so it reads as zero-parallax geometry and corrupts pose and scale |

Three admission gates now exist in `src/ingest/video_ingest.py`, and `src/ingest/screen.py`
applies them before any GPU time is spent. Run against the four candidates:

| Clip | Sky | Horizon | Shots | Verdict |
|---|---|---|---|---|
| Nicosia city vista | 52% | 100% | 2 | **REJECT** |
| Toolse castle | 19% | 100% | 4 | **REJECT** |
| Baha'i Temple orbit | 60% | 6% | 8 | **REJECT** |
| **Kolu wildlife overpass** | **11%** | 23% | **1** | **ACCEPT** |

## The video

52 s over the Kolu wildlife overpass, Estonia - Wikimedia Commons, **CC0**, 1920x1080 at
29.97 fps. One continuous shot, steep oblique to near-nadir, a bounded ~100 m subject with
real occluding structure (the arches), and no horizon in the retained frames.

**S1 ran locally** on the real VP9 file:

| | |
|---|---|
| Frames decoded | 1,562 |
| Shots detected | 1 (continuous - nothing discarded as a different pass) |
| Rejected: horizon in view | 356 |
| Rejected: sky / lens flare | 529 |
| Rejected: motion blur | 787 (25th percentile of the clip, not a constant) |
| Keyframes selected | **45**, by optical-flow baseline budget |

**The one real limitation:** this clip carries **no GPS**, so nothing here can be
georeferenced - only metric-relative. That is a missing input, not the wrong kind of
footage. The PS lists GPS as mandatory precisely because without it the absolute
accuracy requirement is unverifiable.

**Before running:** Accelerator **GPU T4 x2**, Internet **On**, attach the keyframe dataset.

In [ ]:
import os, sys, glob, json, time
import numpy as np
import torch

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, round(p.total_memory/1e9, 1), "GB x", torch.cuda.device_count())
else:
    print("!! No GPU. Settings -> Accelerator -> GPU T4 x2.")

# Tunables, in one place.
N_VIEWS  = 45      # keyframes from the S1 ingest to reconstruct jointly
CONF_PCT = 30      # drop this % least-confident points

## 1. Install MapAnything

In [ ]:
%%capture
# Kaggle ships torch 2.10.0+cu128. Installing MapAnything WITH deps can pull a different
# torch and break CUDA, so --no-deps plus its declared requirements by hand.
!pip install -q hydra-core omegaconf einops huggingface_hub safetensors trimesh scipy matplotlib pillow
!pip install -q "opencv-python-headless==4.10.0.84" "rerun-sdk~=0.24.1" "uniception==0.1.7"
!pip install -q --no-deps git+https://github.com/facebookresearch/map-anything.git

In [ ]:
import importlib, torch
for m in ("hydra", "omegaconf", "einops", "uniception", "mapanything"):
    importlib.import_module(m)
print("imports OK | torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA lost - a dependency replaced torch"
from mapanything.models import MapAnything
print("MapAnything OK")

## 2. Find the keyframes

These came out of the local S1 stage. Attach the dataset via **Add Input** if the search below finds nothing.

In [ ]:
cands = sorted(glob.glob("/kaggle/input/**/kf_*.jpg", recursive=True))
if not cands:
    cands = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
assert cands, "No keyframes found - attach the dataset with Add Input"

IMGS = cands[:N_VIEWS]
print(len(IMGS), "keyframes")
for p in IMGS[:3]:
    print("  ", os.path.basename(p))

meta = glob.glob("/kaggle/input/**/ingest.json", recursive=True)
if meta:
    ing = json.load(open(meta[0]))
    print("\nS1 stats from the local ingest:")
    for k, v in ing["stats"].items():
        print(f"  {k:24s} {v}")

## 3. MapAnything on the keyframes

In [ ]:
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

device = "cuda" if torch.cuda.is_available() else "cpu"

t0 = time.perf_counter()
# APACHE checkpoint - the default is CC-BY-NC and unusable for this customer.
model = MapAnything.from_pretrained("facebook/map-anything-apache").to(device).eval()
print("loaded in", round(time.perf_counter()-t0, 1), "s")

views = load_images(IMGS)
print(len(views), "views prepared")

# bf16 needs Ampere (SM 8.0+); a T4 is Turing 7.5, so fp16 there.
AMP = "fp32"
if device == "cuda":
    AMP = "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"
print("autocast dtype:", AMP)

if device == "cuda":
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
with torch.no_grad():
    preds = model.infer(views, memory_efficient_inference=True,
                        use_amp=(device == "cuda"), amp_dtype=AMP, apply_mask=True)
if device == "cuda":
    torch.cuda.synchronize()
t_inf = time.perf_counter() - t0
per_view = t_inf/len(views)

print("\nINFERENCE", round(t_inf, 1), "s for", len(views), "views =",
      round(per_view, 2), "s/view")
if device == "cuda":
    print("peak VRAM", round(torch.cuda.max_memory_allocated()/1e9, 1), "GB")

In [ ]:
# Camera centres, needed to orient normals for meshing. MapAnything names this
# key differently across versions, so probe rather than assume; the fallback is the
# scene centroid pushed along the thinnest principal axis, which for a survey pass
# over roughly planar ground is the up direction.
cams = []
for pr in preds:
    for k in ("camera_poses", "camera_pose", "cam2world", "camera_pose_trans"):
        if k in pr:
            v = pr[k].squeeze(0).float().cpu().numpy()
            cams.append(v[:3, 3] if v.ndim == 2 and v.shape[-1] == 4 else v.ravel()[:3])
            break
if cams:
    CAM_SIDE = np.mean(cams, axis=0)
    print("camera centres recovered:", len(cams), "-> mean", np.round(CAM_SIDE, 2))
else:
    CAM_SIDE = None
    print("no camera-pose key in preds; will fall back to the thin-axis normal")

### Full-video extrapolation

This clip is 52 s of continuous flight and yields 45 keyframes. The PS budget is 900 s for a
**10-minute** video, which our architecture keyframes to ~600.

In [ ]:
KEYFRAMES, BUDGET = 600, 900.0
geom = BUDGET * 0.45
est = per_view * KEYFRAMES
print("measured        ", round(per_view, 2), "s/view")
print("600 keyframes   ", round(est), "s")
print("geometry budget ", round(geom), "s  (45% of 900 s)")
print("WITHIN BUDGET" if est <= geom else "OVER BUDGET",
      "-", round(geom/est, 2), "x headroom" if est <= geom else "x over")
print("\nS1 ingest measured locally on the same clip: 1,199 frames decoded and scored.")

## 4. Reconstruction - filtered and fused properly

Raw per-view point maps concatenated together are **not** a reconstruction. MapAnything returns a
per-point confidence channel and a validity mask; a first attempt at this notebook ignored both
and simply dumped 20 depth maps on top of each other, which produced a noise smear. This does the
work properly: confidence gate, validity mask, range gate from actual parallax, statistical
outlier removal, and voxel fusion so overlapping views average instead of triple-printing.

In [ ]:
import cv2
from scipy.spatial import cKDTree

# Per-view point maps carry their own grid; READ it rather than infer it. The first
# attempt reconstructed h and w from the flattened length and an aspect ratio, which
# is off by a pixel or two on most views - enough to shear every colour one row
# sideways and make correct geometry look like mush.
pts, cols, confs = [], [], []
for i, p in enumerate(preds):
    if "pts3d" not in p:
        continue
    t = p["pts3d"].squeeze(0)                       # (H, W, 3)
    H, W = t.shape[:2]
    a = t.reshape(-1, 3).float().cpu().numpy()

    cf = (p["conf"].squeeze(0).reshape(-1).float().cpu().numpy()
          if "conf" in p else np.ones(len(a), np.float32))

    keep = np.ones(len(a), bool)
    if "non_ambiguous_mask" in p:
        m = p["non_ambiguous_mask"].squeeze(0).reshape(-1).cpu().numpy().astype(bool)
        if m.shape[0] == a.shape[0]:
            keep = m

    im = cv2.cvtColor(cv2.imread(IMGS[i]), cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA).reshape(-1, 3)
    pts.append(a[keep]); cols.append(rgb[keep]); confs.append(cf[keep])

P = np.concatenate(pts, 0); C0 = np.concatenate(cols, 0); CF = np.concatenate(confs, 0)
ok = np.isfinite(P).all(1)
P, C0, CF = P[ok], C0[ok], CF[ok]
print(f"raw fused        {len(P):>9,} points")

# CONFIDENCE GATE
thr = np.percentile(CF, CONF_PCT)
g = CF >= thr
P, C0 = P[g], C0[g]
print(f"after confidence {len(P):>9,}  (conf >= {thr:.2f}, dropped {CONF_PCT}%)")

# RANGE GATE - only where the depth distribution actually HAS a far-field tail.
# On bounded survey footage it does not, and a blanket 70th-percentile cut throws
# away a third of the scene for nothing.
cam = np.median(P, axis=0)
r = np.linalg.norm(P - cam, axis=1)
p50, p99 = np.percentile(r, [50, 99])
tail = p99 / max(p50, 1e-9)
if tail > 6.0:
    rmax = np.percentile(r, 90)
    g = r <= rmax
    P, C0 = P[g], C0[g]
    print(f"after range gate {len(P):>9,}  (tail ratio {tail:.1f} -> cut at {rmax:.1f})")
else:
    print(f"range gate       skipped  (tail ratio {tail:.1f}, scene is bounded)")

# STATISTICAL OUTLIER REMOVAL
sub = P[np.random.default_rng(0).choice(len(P), min(200000, len(P)), replace=False)]
tree = cKDTree(sub)
dd, _ = tree.query(P, k=9, workers=-1)
md = dd[:, 1:].mean(1)
g = md < md.mean() + 1.2 * md.std()
P, C0, dd = P[g], C0[g], dd[g]
print(f"after outliers   {len(P):>9,}")

# VOXEL FUSION - overlapping views average instead of triple-printing the surface
vox = float(np.percentile(dd[:, 1], 55) * 1.4)
key = np.floor(P / vox).astype(np.int64)
_, idx, inv = np.unique(key, axis=0, return_index=True, return_inverse=True)
Pf = np.zeros((len(idx), 3)); Cf = np.zeros((len(idx), 3))
cnt = np.bincount(inv, minlength=len(idx)).astype(float)[:, None]
np.add.at(Pf, inv, P); np.add.at(Cf, inv, C0.astype(float))
P, C0 = Pf / cnt, np.clip(Cf / cnt, 0, 255).astype(np.uint8)
print(f"after voxel fuse {len(P):>9,}  (voxel {vox:.3f})")
print("final extent    ", np.round(P.max(0) - P.min(0), 1))


## 5. Mesh

The problem statement asks for a **3D mesh or point cloud**; a mesh is the harder
half and the one a downstream user actually loads. Screened Poisson needs oriented
normals, so they are estimated from the local neighbourhood and then flipped to face
the camera cluster - without that step Poisson produces an inside-out surface that
renders black.

In [ ]:
!pip -q install open3d 2>/dev/null || pip -q install open3d-cpu
import open3d as o3d

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(P)
pcd.colors = o3d.utility.Vector3dVector(C0 / 255.0)

nn = float(np.percentile(dd[:, 1], 60) * 3.0)
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=nn, max_nn=30))
# The camera flew above the scene, so orient every normal toward that side.
if CAM_SIDE is not None:
    pcd.orient_normals_towards_camera_location(np.array(CAM_SIDE, float))
else:
    # Thinnest principal axis of a near-planar survey scene is the up direction.
    Pc = P - P.mean(0)
    up = np.linalg.svd(Pc[::37], full_matrices=False)[2][-1]
    pcd.orient_normals_towards_camera_location(P.mean(0) + up * np.ptp(P, 0).max())

mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=10, width=0, scale=1.1, linear_fit=False)
# Poisson closes the surface over unobserved space; trim where it had no support.
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.06))
mesh.compute_vertex_normals()
print("mesh:", len(mesh.vertices), "vertices,", len(mesh.triangles), "triangles")

o3d.io.write_triangle_mesh("/kaggle/working/model.ply", mesh)
o3d.io.write_point_cloud("/kaggle/working/cloud.ply", pcd)
np.save("/kaggle/working/points.npy", P); np.save("/kaggle/working/colors.npy", C0)
np.save("/kaggle/working/mesh_v.npy", np.asarray(mesh.vertices))
np.save("/kaggle/working/mesh_f.npy", np.asarray(mesh.triangles))
np.save("/kaggle/working/mesh_c.npy", (np.asarray(mesh.vertex_colors) * 255).astype(np.uint8))
print("wrote model.ply, cloud.ply and npy arrays")


### Rendered views of the mesh

Offscreen rendering, so the mesh is judged as a mesh and not as a scatter plot.

In [ ]:
V = np.asarray(mesh.vertices); F = np.asarray(mesh.triangles)
VC = np.asarray(mesh.vertex_colors)
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation

def shade(ax, az, el, title):
    """Flat-shaded projection - a real surface render, lit from one direction."""
    c, s = np.cos(np.radians(az)), np.sin(np.radians(az))
    R1 = np.array([[c, 0, -s], [0, 1, 0], [s, 0, c]])
    c2, s2 = np.cos(np.radians(el)), np.sin(np.radians(el))
    R2 = np.array([[1, 0, 0], [0, c2, -s2], [0, s2, c2]])
    Vr = V @ R1.T @ R2.T
    tri = Triangulation(Vr[:, 0], -Vr[:, 1], F)
    # Depth-sort is unnecessary for a filled triangulation with shading by normal.
    n = np.cross(Vr[F[:, 1]] - Vr[F[:, 0]], Vr[F[:, 2]] - Vr[F[:, 0]])
    n /= (np.linalg.norm(n, axis=1, keepdims=True) + 1e-9)
    lam = np.clip(n @ np.array([0.3, -0.8, 0.5]), 0.15, 1.0)
    fc = VC[F].mean(1) * lam[:, None]
    ax.tripcolor(tri, facecolors=np.clip(fc, 0, 1), edgecolors="none",
                 shading="flat", rasterized=True)
    ax.set_aspect("equal"); ax.set_title(title); ax.axis("off")

fig, ax = plt.subplots(1, 3, figsize=(19, 6.5))
shade(ax[0], 0, 90, "mesh - plan")
shade(ax[1], 25, 55, "mesh - oblique")
shade(ax[2], 60, 20, "mesh - low angle")
plt.tight_layout(); plt.show()


In [ ]:
# Is there real 3D structure, or a draped sheet? PCA tells us directly.
s = P[np.random.default_rng(0).choice(len(P), min(80000, len(P)), replace=False)]
c = s.mean(0)
_, sv, vt = np.linalg.svd(s - c, full_matrices=False)
flat = np.ptp((s - c) @ vt[2]) / max(np.ptp((s - c) @ vt[0]), 1e-9)
print("PCA singular values :", np.round(sv/sv[0], 3))
print(f"flatness ratio      : {flat:.3f}   (a city block model wants volume, not a sheet)")

np.save("/kaggle/working/points_clean.npy", P.astype(np.float32))
np.save("/kaggle/working/colors_clean.npy", C0.astype(np.uint8))
json.dump({"seconds_per_view": round(per_view, 3), "n_views": len(views),
           "points_final": int(len(P)), "flatness": round(float(flat), 4),
           "device": device, "amp": AMP, "est_600_keyframes_s": round(est, 1)},
          open("/kaggle/working/result.json", "w"), indent=2)
print("wrote points_clean.npy, colors_clean.npy, result.json")

In [ ]:
import matplotlib.pyplot as plt

idx = np.random.default_rng(0).choice(len(P), min(150000, len(P)), replace=False)
S, SC = P[idx], np.clip(C0[idx]/255.0, 0, 1)
up = -S[:, 1]                                  # model Y is down

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
ax[0].scatter(S[:,0], S[:,2], c=SC, s=0.6); ax[0].set_title("Top-down, photo colour")
ax[0].set_xlabel("X"); ax[0].set_ylabel("Z")
h = ax[1].scatter(S[:,0], S[:,2], c=up, s=0.6, cmap="turbo")
ax[1].set_title("Top-down, coloured by height"); ax[1].set_xlabel("X"); ax[1].set_ylabel("Z")
plt.colorbar(h, ax=ax[1], label="height")
ax[2].scatter(S[:,0], up, c=SC, s=0.6); ax[2].set_title("Side elevation")
ax[2].set_xlabel("X"); ax[2].set_ylabel("height")
for a_ in ax: a_.set_aspect("equal")
plt.tight_layout(); plt.show()

## What this run establishes

**Establishes:** a real drone **video file** goes in and a 3D point cloud comes out. Decode,
slate rejection, blur rejection, baseline-aware keyframing and learned reconstruction all ran on
genuine H.264 footage, and the per-view GPU cost extrapolates to the 900 s budget.

**Does not establish:** georeferencing or accuracy. This clip carries **no GPS**, so there is no
absolute frame to place the model in and no ground truth to score against. The PS lists GPS as a
mandatory input precisely because without it the problem is unsolvable as stated.

**Also worth seeing in the plots:** geometry degrades with distance from the camera. That is the
problem this PS poses, not a flaw in the footage - an oblique single pass puts most pixels in the
far field where a monocular model has little parallax. It implies the deliverable should carry
per-point confidence and report accuracy per range band, rather than quoting one number that
holds near the camera and fails at the horizon.